# **FiQA-2018 Information Retrieval System**

---

This notebook implements and evaluates a comparative information retrieval system on the
FiQA-2018 financial question-answering dataset (Thakur et al., 2021).

**Retrieval pipeline:**

| Section | Description |
|---|---|
| 1 | Dataset loading and inspection |
| 2 | BM25 baseline — lexical retrieval |
| 3 | BM25 parameter tuning — grid search over $k_1$ and $b$ |
| 4 | SBERT dense retrieval — semantic embedding with FAISS |
| 5 | Results summary — metric comparison across models |
| 6 | Interactive search GUI |

> **Reproducibility:** This notebook is fully self-contained. All data downloads automatically at runtime. Open in Colab, set the runtime to GPU (T4).


In [1]:
%%capture
!pip install --upgrade beir
!pip install faiss-cpu bm25s sentence-transformers gradio

import os, logging, warnings, itertools
import numpy as np
import pandas as pd

# Suppresses debug output from bm25s
root_logger = logging.getLogger()
root_logger.setLevel(logging.WARNING)

for handler in root_logger.handlers:
    handler.setLevel(logging.WARNING)
logging.getLogger('bm25s').setLevel(logging.WARNING)

# Suppresses HuggingFace Hub and tqdm warnings
warnings.filterwarnings('ignore', category=UserWarning, module='huggingface_hub')
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [11]:
import nbformat

NOTEBOOK_PATH = "/content/drive/MyDrive/Colab Notebooks/fiqa_ir_system_final.ipynb"

with open(NOTEBOOK_PATH, "r") as f:
    nb = nbformat.read(f, as_version=4)

for cell in nb.cells:
    if "widgets" in cell.get("metadata", {}):
        del cell["metadata"]["widgets"]

if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

with open(NOTEBOOK_PATH, "w") as f:
    nbformat.write(nb, f)

print("Done. Notebook cleaned and saved back to Drive.")

Done. Notebook cleaned and saved back to Drive.


## Section 1 — Loading the Dataset: FiQA-2018

---



FiQA-2018 (Financial Question Answering) is a dataset originally drawn from financial community platforms and later standardised within the BEIR benchmark (Thakur et al., 2021). It consists of natural language financial questions paired with relevant answers, making it well-suited for evaluating retrieval models in which questions and answers rarely share exact vocabulary, which challenges purely lexical systems.

BEIR provides data in three components:

| Component | Description | Size (test split) |
|:---|:---|:---|
| **corpus** | Document passages to be indexed and retrieved from | 57,638 documents |
| **queries** | Financial questions submitted to the system | 648 queries |
| **qrels** | Ground-truth relevance judgements | 648 judgements |

The dataset downloads automatically from the BEIR public repository.


In [ ]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader

# Downloads and unzips FiQA-2018 from the BEIR repository
url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip"
data_path = util.download_and_unzip(url, "/content")

# Loads the test split
# qrels are only available for test
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

print(f"Corpus:  {len(corpus):,} documents")
print(f"Queries: {len(queries):,} queries")
print(f"Qrels:   {len(qrels):,} relevance judgements")

### 1.1 Data Inspection

Before building the retrieval system we examine a sample of the corpus, queries, and relevance judgements. This is to confirm the data loaded correctly and informs us about document length, vocabulary, and the nature of the retrieval task.

In [ ]:
# Sample document, query, and relevance judgement
sample_doc_id = list(corpus.keys())[0]
sample_qid    = list(queries.keys())[0]

print("SAMPLE DOCUMENT")
print(f"  ID:    {sample_doc_id}")
print(f"  Title: {corpus[sample_doc_id].get('title', '(none)')}")
print(f"  Text:  {corpus[sample_doc_id]['text'][:200]}...\n")

print("SAMPLE QUERY")
print(f"  ID:   {sample_qid}")
print(f"  Text: {queries[sample_qid]}\n")

print("RELEVANT DOCUMENTS (qrel)")
for doc_id, score in list(qrels[sample_qid].items())[:3]:
    print(f"  doc_id={doc_id}, relevance={score}")

## Section 2 — BM25 Baseline

---



BM25 (*Best Matching 25*) is the standard lexical retrieval model. It scores each document $d$ against query $q$ using the Okapi BM25 formula (Robertson & Zaragoza, 2009):

$$\text{BM25}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t,d) \cdot (k_1 + 1)}{f(t,d) + k_1\left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}$$

**Parameters:**

- $f(t, d)$ — term frequency of $t$ in $d$
- $\text{IDF}(t)$ — inverse document frequency, down-weighting common terms
- $|d|$ — document length; $\text{avgdl}$ — mean document length in the corpus
- $k_1$ — term frequency saturation (higher = slower saturation)
- $b$ — length normalisation strength (0 = none, 1 = full)

We index the corpus and retrieve the top-100 documents per query, then evaluate using BEIR's `EvaluateRetrieval`. Initial parameters follow Robertson & Zaragoza (2009) defaults: $k_1 = 0.9$, $b = 0.4$.

The retrieval pipeline:

1. Concatenates document title and body into a single string per document
2. Tokenises corpus and builds the BM25 index
3. Tokenises all 648 test queries and retrieves the top-100 documents per query
4. Converts results to BEIR format and evaluates with `EvaluateRetrieval`

BM25 rewards documents that contain query terms frequently
but penalises very long documents that match simply by virtue of their length.
Term frequency contribution saturates- a term appearing 10 times adds
proportionally less than one appearing twice, controlled by $k_1$.

In [ ]:
import bm25s
from beir.retrieval.evaluation import EvaluateRetrieval

# Flatten corpus dictionary into parallel lists of IDs and text strings
# title and body text are concatenated so both fields contribute to BM25 scoring
corpus_ids   = list(corpus.keys())
corpus_texts = [
    (corpus[doc_id].get("title", "") + " " + corpus[doc_id]["text"]).strip()
    for doc_id in corpus_ids
]

# Tokenises corpus and builds BM25 index with default hyperparameters
corpus_tokens = bm25s.tokenize(corpus_texts, stopwords=None, stemmer=None)
retriever     = bm25s.BM25(k1=0.9, b=0.4)
retriever.index(corpus_tokens)

# Tokenises queries using identical settings to the corpus (no stopwords/stemming)
# retrieves the top 100 documents for each of the 648 queries
query_ids    = list(queries.keys())
query_texts  = list(queries.values())
query_tokens = bm25s.tokenize(query_texts, stopwords=None, stemmer=None)
results_idx, scores = retriever.retrieve(query_tokens, k=100)

# Converts raw to BEIR format
bm25_results = {
    query_ids[i]: {
        corpus_ids[results_idx[i, j]]: float(scores[i, j])
        for j in range(results_idx.shape[1])
    }
    for i in range(len(query_ids))
}

# Evaluates BM25 retrieval results against ground-truth qrels
ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(
    qrels, bm25_results, [10, 100]
)

print("BM25 Baseline  (k1=0.9, b=0.4)\n")
print(f"  NDCG@10:    {ndcg['NDCG@10']:.4f}")
print(f"  MAP@100:    {_map['MAP@100']:.4f}")
print(f"  Recall@100: {recall['Recall@100']:.4f}")
print(f"  P@10:       {precision['P@10']:.4f}")

**NDCG@10** — measures ranking quality, rewards putting the most relevant docs at the very top. A score of $23.45$% is in line with published BM25 baselines especially as FiQA is a hard dataset (financial Q&A).

**MAP@100** — mean average precision, measures overall ranking quality across all relevant docs up to rank 100. A score of $18.74$% - MAP is stricter than NDCG@10 because it accounts for all relevant documents not just the top.

**Recall@100** — what fraction of all relevant documents were retrieved in the top 100. In this case, $49.52$% of all relevant documents are captured which is expected for a pure lexical method with no semantic understanding.

**P@10** — how many out of the top 10 results were actually relevant. A $0.648$% is relevant per query which is because the dataset has very few relevant documents per query (sparse qrels).

## Section 3 — BM25 Parameter Tuning

---



The default BM25 parameters are for general purposes. Tuning them to the specific characteristics of FiQA-2018, its document length distribution, domain vocabulary, and query style, can produce meaningful improvements in performance.

We perform a grid search over two parameters:

- $k_1 \in \{0.9,\ 1.2,\ 1.5\}$ — low to high term frequency saturation
- $b \in \{0.5,\ 0.75,\ 0.9\}$ — moderate to aggressive length normalisation

All 9 combinations are evaluated using **NDCG@10** as the selection criterion, giving  $3×3=9$  retrieval runs. The best configuration is then re-evaluated on the full set of metrics.

> **Methodological note:** FiQA-2018 provides qrels only for the test split. Ideally, tuning would use a separate validation set. Since none is available, tuning and testing occur on the same split. This is a known limitation acknowledged in the experimental design.

In [ ]:
# k1 controls term frequency saturation
# higher values give more weight to repeated terms before the score plateaus
# b controls document length normalisation, higher values penalise longer docs more
tuning_results = []

# NDCG@10 is the selection criterioN
# Evaluate all 9 (k1, b) combinations
# for each configuration: rebuild the index, retrieve top-100, evaluate NDCG@10
for k1, b in itertools.product([0.9, 1.2, 1.5], [0.5, 0.75, 0.9]):
    r = bm25s.BM25(k1=k1, b=b)
    r.index(corpus_tokens)
    idx, s = r.retrieve(query_tokens, k=100)
    run = {
        query_ids[i]: {corpus_ids[idx[i,j]]: float(s[i,j]) for j in range(100)}
        for i in range(len(query_ids))
    }

    # Evaluate and record
    n, m, _, _ = EvaluateRetrieval.evaluate(qrels, run, [10])
    tuning_results.append({"k1": k1, "b": b,
                            "NDCG@10": round(n["NDCG@10"], 4),
                            "MAP@10":  round(m["MAP@10"],  4)})
    print(f"  k1={k1}, b={b}:  NDCG@10={n['NDCG@10']:.4f}")

# Sort all configurations by NDCG@10 descending, best config appears at the top
df_tuning = pd.DataFrame(tuning_results).sort_values("NDCG@10", ascending=False)
display(df_tuning)

### 3.1 Best Configuration

The grid search identifies $k_1 = 0.9$, $b = 0.75$ as the optimal configuration. The increased $b$ value (0.4 to 0.75) applies stronger length normalisation, which helps on FiQA-2018 where document lengths vary considerably. While the improvements are modest in absolute terms, they are consistent across all metrics, suggesting the tuning is meaningful rather than incidental. The tuned configuration (k1=0.9, b=0.75) is carried forward as the BM25 baseline for the hybrid pipeline. We re-index and re-evaluate with this configuration to obtain the full set of metrics, and use the revamped values to quantify the improvement over the baseline.


In [ ]:
# Re-index with the best configuration identified above
best_retriever = bm25s.BM25(k1=0.9, b=0.75)
best_retriever.index(corpus_tokens)
idx, s = best_retriever.retrieve(query_tokens, k=100)

bm25_tuned_results = {
    query_ids[i]: {corpus_ids[idx[i,j]]: float(s[i,j]) for j in range(100)}
    for i in range(len(query_ids))
}

ndcg_t, map_t, recall_t, prec_t = EvaluateRetrieval.evaluate(
    qrels, bm25_tuned_results, [10, 100]
)

print("BM25 Tuned  (k1=0.9, b=0.75)\n")
print(f"  NDCG@10:    {ndcg_t['NDCG@10']:.4f}  (Increase: {ndcg_t['NDCG@10'] - ndcg['NDCG@10']:+.4f})")
print(f"  MAP@100:    {map_t['MAP@100']:.4f}  (Increase: {map_t['MAP@100']  - _map['MAP@100']:+.4f})")
print(f"  Recall@100: {recall_t['Recall@100']:.4f}  (Increase: {recall_t['Recall@100'] - recall['Recall@100']:+.4f})")
print(f"  P@10:       {prec_t['P@10']:.4f}  (Increase: {prec_t['P@10'] - precision['P@10']:+.4f})")

## Section 4 — Dense Retrieval with SBERT

---



BM25 matches query and document terms literally. It cannot recognise that *"portfolio diversification"* and *"spreading investment risk"* express the same idea. This is the **vocabulary mismatch problem**, and it is particularly pronounced in FiQA-2018, where questions are rarely phrased using the exact words of the answer passages.

Dense retrieval resolves this by encoding both queries and documents as dense vectors in a shared semantic space using a pre-trained language model. Documents with similar meaning cluster together regardless of their form, enabling retrieval based on conceptual proximity rather than term overlap.

**Model:** `multi-qa-mpnet-base-dot-v1` (Reimers & Gurevych, 2019)

- Fine-tuned on diverse QA datasets (MS MARCO, Natural Questions, etc.) — well-calibrated for question-to-passage retrieval
- Produces 768-dimensional L2-normalised vectors
- Dot-product similarity equals cosine similarity on normalised vectors
- fp16 (half precision) to reduce VRAM usage by 50%

This model was selected over alternatives (e.g. `all-mpnet-base-v2`) because it was fine-tuned specifically for asymmetric question-to-passage retrieval which is the exact structure of the FiQA-2018 task.

### 4.1 Corpus Encoding

Every document in the 57,638-document corpus is encoded into a 768-dimensional vector. With `normalize_embeddings=True`, all vectors are L2-normalised, so dot-product similarity is equivalent to cosine similarity, a requirement for correct operation of the FAISS `IndexFlatIP` index built in the next step. This encoding step runs once and takes approximately **4 minutes on a T4 GPU**.

In [ ]:
import os, torch, gc, warnings
from sentence_transformers import SentenceTransformer

# Reduces fragmentation that can cause OOM errors even when total free VRAM is sufficient
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
warnings.filterwarnings('ignore')  # suppresses HF Hub and model load warnings

# Ensuring maximum available VRAM for encoding
gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
free_gb = torch.cuda.mem_get_info()[0] / 1024**3 if device == "cuda" else 0
print(f"Device: {device}  |  VRAM free before model load: {free_gb:.1f} GB")

sbert_model = SentenceTransformer("multi-qa-mpnet-base-dot-v1", device=device)

# Casts to fp16 to halve tensor memory to prevent memory overload
if device == "cuda":
    sbert_model = sbert_model.half()

free_after = torch.cuda.mem_get_info()[0] / 1024**3 if device == "cuda" else 0
print(f"VRAM free after model load:  {free_after:.1f} GB")

# Encode all corpus documents into 768-dimensional dense vectors
# normalize_embeddings=True applies L2 normalisation so dot-product
# similarity equals cosine similarity as required for FAISS inner-product search
corpus_embed = sbert_model.encode(
    corpus_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(f"Embedding matrix: {corpus_embed.shape}")


### 4.2 FAISS Index

With 57,638 document vectors computed, we need an efficient way to find the top-100 nearest neighbours for each query vector. **FAISS** (Johnson et al., 2019) provides optimised vector similarity search. We use `IndexFlatIP`:

| Property | Detail |
|:---|:---|
| `Flat` | Exact search — no approximation, guaranteed optimal results |
| `IP` | Inner Product similarity — correct for L2-normalised vectors |
| Scalability | Appropriate for this corpus size (~57K docs); for >1M docs, `IndexIVFFlat` (approximate) is preferred |

Once built, the index is queried using the encoded query vectors to retrieve the top-100 documents per query.


In [ ]:
import faiss

# Builds FAISS exact inner-product index
index = faiss.IndexFlatIP(corpus_embed.shape[1])
index.add(corpus_embed)
print(f"FAISS index: {index.ntotal:,} vectors, dimension {corpus_embed.shape[1]}")

# Encodes queries with the same model and normalisation
query_embed = sbert_model.encode(
    query_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Retrieves top-100 per query
sbert_scores, sbert_idx = index.search(query_embed, k=100)

# Builds BEIR-compatible results dict
sbert_results = {
    query_ids[i]: {
        corpus_ids[sbert_idx[i, j]]: float(sbert_scores[i, j])
        for j in range(100)
    }
    for i in range(len(query_ids))
}

# Evaluates
sbert_ndcg, sbert_map, sbert_recall, sbert_prec = EvaluateRetrieval.evaluate(
    qrels, sbert_results, [10, 100]
)

print("\nSBERT Dense Retrieval Results:\n")
print(f"  NDCG@10:    {sbert_ndcg['NDCG@10']:.4f}")
print(f"  MAP@100:    {sbert_map['MAP@100']:.4f}")
print(f"  Recall@100: {sbert_recall['Recall@100']:.4f}")
print(f"  P@10:       {sbert_prec['P@10']:.4f}")

## Section 5 — Results Summary

---



All three configurations are evaluated on the FiQA-2018 test split (648 queries) using four standard retrieval metrics.

| Metric | What it measures |
|---|---|
| **NDCG@10** | Ranking quality in the top 10 results (primary metric) |
| **MAP@100** | Mean average precision across the top 100 |
| **Recall@100** | Fraction of all relevant docs found in the top 100 |
| **P@10** | Precision — fraction of top 10 results that are relevant |

In [ ]:
df_results = pd.DataFrame([
    {"Model":       "BM25 Baseline",
     "k1": 0.9, "b": 0.40,
     "NDCG@10":    round(ndcg['NDCG@10'],        4),
     "MAP@100":    round(_map['MAP@100'],          4),
     "Recall@100": round(recall['Recall@100'],     4),
     "P@10":       round(precision['P@10'],        4)},
    {"Model":       "BM25 Tuned",
     "k1": 0.9, "b": 0.75,
     "NDCG@10":    round(ndcg_t['NDCG@10'],       4),
     "MAP@100":    round(map_t['MAP@100'],          4),
     "Recall@100": round(recall_t['Recall@100'],   4),
     "P@10":       round(prec_t['P@10'],           4)},
    {"Model":       "SBERT Dense",
     "k1": None, "b": None,
     "NDCG@10":    round(sbert_ndcg['NDCG@10'],   4),
     "MAP@100":    round(sbert_map['MAP@100'],      4),
     "Recall@100": round(sbert_recall['Recall@100'],4),
     "P@10":       round(sbert_prec['P@10'],       4)},
])

display(df_results)

### Discussion

---



**BM25 tuning** generates a small but consistent improvement across all metrics (NDCG@10: 0.2345 to 0.2401, increment: +0.0056, +2.4%). The optimal $b = 0.75$ is higher than the default 0.4 and applies stronger length normalisation, which helps on FiQA-2018 where document passage lengths vary considerably. The best $k_1$ remains 0.9, suggesting that term frequency saturation is already well-calibrated at the default value for this dataset.

**SBERT** achieves nearly double BM25's NDCG@10 (0.4443 against 0.2401, +84.6%). This gap is explained by how financial questions on community platforms are phrased in natural conversational language, while the answer passages use financial terminology. Whereas dense semantic matching bridges this vocabulary gap, BM25's lexical overlap cannot.

**Recall@100** improves from 0.508 (BM25 Tuned) to 0.794 (SBERT), meaning the dense
model recovers approximately 56% more relevant documents within the top 100. This
demonstrates the core limitation of lexical retrieval on domain-specific QA datasets. BM25 structurally misses relevant documents that do not share the specific terms with
the query, regardless of parameter tuning.

Combined together, these results demonstrate that the choice of retrieval framework
matters far more than parameter optimisation within a paradigm. The +2.4% gain from
BM25 tuning is overshadowed by the +84.6% gain from switching to semantic retrieval,
highlighting the importance of model selection for domain-specific IR tasks.


## Section 6 — Interactive Search Interface

---



The Gradio interface below is a live demonstration of both retrieval models
built in this notebook. It uses the indices and embeddings already computed so no
re-indexing is required.

**Interface features:**

| Feature | Description |
|:---|:---|
| Query input | Free-text search box; supports Enter-to-search |
| Model selector | Switch between **BM25 (Tuned)** (lexical) and **SBERT Dense** (semantic) |
| Top-K slider | Adjustable result count from 1 to 20 |
| Results table | Rank, Document ID, relevance score, title, and 250-character snippet |
| Example queries | Pre-loaded queries illustrating both retrieval modes |
| Public URL | `share=True` generates a shareable Gradio link for the demo recording |


In [ ]:
import gradio as gr


def search(query: str, model: str, top_k: int) -> pd.DataFrame:
    """Retrieve top-k documents for a query using the selected model."""
    if not query.strip():
        return pd.DataFrame(columns=["Rank", "Doc ID", "Score", "Title", "Snippet"])

    if model == "BM25 (Tuned)":
        tokens         = bm25s.tokenize([query], stopwords=None, stemmer=None)
        idx, scores_q  = best_retriever.retrieve(tokens, k=top_k)
        ranked         = [(corpus_ids[idx[0, j]], float(scores_q[0, j])) for j in range(top_k)]
    else:  # SBERT Dense
        q_embed        = sbert_model.encode([query], normalize_embeddings=True,
                                             convert_to_numpy=True)
        scores_q, idxs = index.search(q_embed, k=top_k)
        ranked         = [(corpus_ids[idxs[0, j]], float(scores_q[0, j])) for j in range(top_k)]

    rows = []
    for rank, (doc_id, score) in enumerate(ranked, 1):
        doc     = corpus.get(doc_id, {})
        title   = (doc.get("title") or "").strip()
        snippet = doc.get("text", "")[:250].replace("\n", " ").strip()
        rows.append({"Rank": rank, "Doc ID": doc_id,
                     "Score": round(score, 4), "Title": title, "Snippet": snippet})

    return pd.DataFrame(rows)


with gr.Blocks(title="FiQA-2018 Search Engine", theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        """# 🔍 FiQA-2018 Search Engine
**ECS736P/U — Information Retrieval | Group 26**

Search 57,638 financial documents using BM25 (lexical) or SBERT (semantic) retrieval."""
    )

    with gr.Row():
        query_box = gr.Textbox(
            label="Search Query",
            placeholder="e.g. How do I diversify my investment portfolio?",
            lines=1, scale=4
        )
        model_dd = gr.Dropdown(
            choices=["BM25 (Tuned)", "SBERT Dense"],
            value="SBERT Dense",
            label="Retrieval Model", scale=1
        )
        topk_sl = gr.Slider(1, 20, value=10, step=1, label="Top-K", scale=1)

    search_btn = gr.Button("Search", variant="primary")

    results_df = gr.Dataframe(
        headers=["Rank", "Doc ID", "Score", "Title", "Snippet"],
        label="Retrieved Documents",
        interactive=False,
        wrap=True
    )

    gr.Examples(
        examples=[
            ["How do I diversify my investment portfolio?",       "SBERT Dense",  10],
            ["What is the difference between stocks and bonds?",  "SBERT Dense",  10],
            ["capital gains tax rate long term investments",      "BM25 (Tuned)", 10],
            ["how to open a Roth IRA account",                    "BM25 (Tuned)", 10],
        ],
        inputs=[query_box, model_dd, topk_sl]
    )

    search_btn.click(fn=search, inputs=[query_box, model_dd, topk_sl], outputs=results_df)
    query_box.submit(fn=search, inputs=[query_box, model_dd, topk_sl], outputs=results_df)


demo.launch(share=True)


## References

---



Johnson, J., Douze, M. and Jégou, H. (2019) 'Billion-Scale Similarity Search with GPUs', *IEEE Transactions on Big Data*, 7(3), pp. 535–547. Available at: https://doi.org/10.1109/TBDATA.2019.2921572

Lassance, C., Déjean, H. and Clinchant, S. (2024) 'BM25S: Orders of Magnitude Faster Lexical Search via Eager Sparse Scores', *arXiv preprint arXiv:2407.03618*. Available at: https://arxiv.org/abs/2407.03618

Reimers, N. and Gurevych, I. (2019) 'Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks', in *Proceedings of the 2019 Conference on Empirical Methods in Natural Language Processing (EMNLP)*. Hong Kong: Association for Computational Linguistics, pp. 3982–3992. Available at: https://doi.org/10.18653/v1/D19-1410

Robertson, S. and Zaragoza, H. (2009) 'The Probabilistic Relevance Framework: BM25 and Beyond', *Foundations and Trends in Information Retrieval*, 3(4), pp. 333–389. Available at: https://doi.org/10.1561/1500000019

Thakur, N., Reimers, N., Rücklé, A., Srivastava, A. and Gurevych, I. (2021) 'BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models', in *Proceedings of the Neural Information Processing Systems Track on Datasets and Benchmarks (NeurIPS Datasets and Benchmarks 2021)*. Available at: https://arxiv.org/abs/2104.08663
